In [15]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Entropy Function

In [2]:
def my_entropy(input_image):

    arr = input_image.flatten().astype(np.int64)

    if arr.min() != 1:
        arr = arr - arr.min() + 1

    p = np.zeros(arr.max(), dtype=np.float64)
    for v in arr:
        p[v - 1] += 1

    p = p / p.sum()
    p = p[p != 0]
    entropy = np.sum(-p * np.log2(p))

    return entropy

## PSNR Function

In [53]:
def peak_signal_noise_ratio(image1: np.ndarray, image2: np.ndarray):
    if image1.shape != image2.shape:
        print('Input images don’t have the same shape')
        return -1

    max_value = max(image1.max(), image2.max())
    max_value_square = max_value ** 2

    # mean-square-error
    img1_flat = image1.flatten()
    img2_flat = image2.flatten()

    error = img1_flat - img2_flat
    error_square = error ** 2
    error_square_sum = np.sum(error_square)
    mean_error_square_sum = error_square_sum / len(img1_flat)

    if mean_error_square_sum == 0:
        return np.inf
    return 10 * np.log10(max_value_square / (mean_error_square_sum))

# Question1: Median Edge Predictor (MED)

In [54]:
def med_predictor(input_image):
    input_image = input_image.astype(np.int16)
    H, W = input_image.shape

    error_image = np.zeros((H, W), dtype=np.int16)

    padded_Image = np.pad(input_image, ((1, 0), (1, 0)), mode='constant', constant_values=0)

    for i in range(1, H + 1):
        for j in range(1, W + 1):

            a = padded_Image[i, j - 1]
            b = padded_Image[i - 1, j]
            c = padded_Image[i - 1, j - 1]

            if c >= max(a, b):
                x = min(a, b)
            elif c <= min(a, b):
                x = max(a, b)
            else:
                x = a + b - c

            error_image[i - 1, j - 1] = input_image[i - 1, j - 1] - x

    return error_image


def med_reconstructor(error_image):
    error_image = error_image.astype(np.int16)
    H, W = error_image.shape

    prediction = np.zeros((H + 1, W + 1), dtype=np.int16)
    reconstructed_image = np.zeros((H, W), dtype=np.uint8)

    for i in range(1, H + 1):
        for j in range(1, W + 1):
            a = prediction[i, j - 1]
            b = prediction[i - 1, j]
            c = prediction[i - 1, j - 1]

            if c >= max(a, b):
                x = min(a, b)
            elif c <= min(a, b):
                x = max(a, b)
            else:
                x = a + b - c

            prediction[i, j] = x + error_image[i - 1, j - 1]
            reconstructed_image[i - 1, j - 1] = np.uint8(prediction[i, j])

    return reconstructed_image

## MED Evaluation

In [55]:
test_path = ['Images/CLIC_2025_1.png', 'Images/CLIC_2025_2.png', 'Images/Kodak_01.png', 'Images/Kodak_23.png',
             'Images/Livingroom.tif', 'Images/Bridge.tif', 'Images/Baboon.tif', 'Images/Peppers.bmp',
             'Images/MRI_1.tif', 'Images/MRI_2.tif', 'Images/Retina.tif', 'Images/Cells.png']

evaluation_table_med = []
for image_name in test_path:
    name = image_name.split('/')[1].split('.')[0]
    image = cv2.imread(image_name, cv2.IMREAD_UNCHANGED)

    med_error = med_predictor(image)
    med_reconstruct = med_reconstructor(med_error)

    evaluation_table_med.append({'Image Name': name,'Image Entropy': my_entropy(image),
                             'Error Entropy(MED)': my_entropy(med_error), 'PSNR (initial-reconstruct)': peak_signal_noise_ratio(image, med_reconstruct)})

evaluation_table_med = pd.DataFrame(evaluation_table_med)
evaluation_table_med

,Image Name,Image Entropy,Error Entropy(MED),PSNR (initial-reconstruct)
0,CLIC_2025_1,7.573678,4.024232,inf
1,CLIC_2025_2,7.615759,5.889772,inf
2,Kodak_01,7.161006,5.511179,inf
3,Kodak_23,7.251587,3.829204,inf
4,Livingroom,7.295174,4.839180,inf
5,Bridge,7.683018,5.668946,inf
6,Baboon,7.292549,5.233969,inf
7,Peppers,7.571478,4.843694,inf
8,MRI_1,6.428016,3.765981,inf
9,MRI_2,6.619796,3.612304,inf


# Question2: Third Order Linear Predictor (Optimum Mode)

## Calculate Coefficients

In [56]:
def compute_third_order_coef(img):

    img = img.astype(np.float64)

    a = img[1:, :-1].flatten()
    b = img[:-1, 1:].flatten()
    c = img[:-1, :-1].flatten()
    x = img[1:, 1:].flatten()

    A = np.sum(a*a)
    B = np.sum(b*b)
    C = np.sum(c*c)
    D = np.sum(a*b)
    E = np.sum(a*c)
    F = np.sum(b*c)

    Xa = np.sum(a*x)
    Xb = np.sum(b*x)
    Xc = np.sum(c*x)

    M = np.array([
        [A, D, E, 1],
        [D, B, F, 1],
        [E, F, C, 1],
        [1, 1, 1, 0]
    ], dtype=np.float64)

    rhs = np.array([Xa, Xb, Xc, 1], dtype=np.float64)
    sol = np.linalg.solve(M, rhs)

    alpha, beta = sol[0], sol[1]
    return alpha, beta

## Three Optimum Predictor

In [57]:
def three_optimum_predictor(input_image):
    input_image = input_image.astype(dtype=np.float64)
    H, W = input_image.shape

    error_image = np.zeros((H, W), dtype=np.float64)
    alpha, beta = compute_third_order_coef(input_image)
    gamma = 1 - (alpha + beta)

    padded_Image = np.pad(input_image, ((1, 0), (1, 0)), mode='constant', constant_values=0)

    for i in range(1, H + 1):
        for j in range(1, W + 1):

            a = padded_Image[i, j - 1]
            b = padded_Image[i - 1, j]
            c = padded_Image[i - 1, j - 1]
            x = (alpha* a) + (beta*b) + (gamma*c)

            error_image[i - 1, j - 1] = input_image[i - 1, j - 1] - x

    return error_image, alpha, beta


def three_optimum_reconstructor(error_image, alpha, beta):
    error_image = error_image.astype(np.float64)
    H, W = error_image.shape
    gamma = 1 - (alpha + beta)

    prediction = np.zeros((H + 1, W + 1), dtype=np.float64)
    reconstructed_image = np.zeros((H, W), dtype=np.float64)

    for i in range(1, H + 1):
        for j in range(1, W + 1):
            a = prediction[i, j - 1]
            b = prediction[i - 1, j]
            c = prediction[i - 1, j - 1]

            x = (alpha*a) + (beta*b) + (gamma*c)

            prediction[i, j] = x + error_image[i - 1, j - 1]
            reconstructed_image[i - 1, j - 1] = prediction[i, j]

    return reconstructed_image

## Evaluation

In [58]:
evaluation_table_three_order = []
for image_name in test_path:
    name = image_name.split('/')[1].split('.')[0]
    image = cv2.imread(image_name, cv2.IMREAD_UNCHANGED)

    three_opt_error, alpha, beta = three_optimum_predictor(image)
    three_opt_reconstruct = three_optimum_reconstructor(three_opt_error, alpha, beta)

    evaluation_table_three_order.append({'Image Name': name, 'Error Entropy(OPT-3)': my_entropy(three_opt_error),
                                         'PSNR (initial-reconstruct)': peak_signal_noise_ratio(image, three_opt_reconstruct)})

evaluation_table_three_order = pd.DataFrame(evaluation_table_three_order)
evaluation_table_three_order

,Image Name,Error Entropy(OPT-3),PSNR (initial-reconstruct)
0,CLIC_2025_1,3.572794,381.918793
1,CLIC_2025_2,5.739044,inf
2,Kodak_01,5.511926,inf
3,Kodak_23,3.543218,inf
4,Livingroom,4.830203,310.046808
5,Bridge,5.599623,378.240748
6,Baboon,5.015353,inf
7,Peppers,4.712849,376.289565
8,MRI_1,3.204443,285.439082
9,MRI_2,2.975721,307.270460
